# 👕 AI Product Intelligence System

## Task 2: Unique Product Catalog Creation

**Submitted By:** Ede Dinesh Venkata Pavan Kumar

ACE Engineering College

GenAI Bootcamp – Day 2

# 📌 Problem Statement

Large e-commerce platforms often contain duplicate or highly similar products uploaded by different sellers.

The objective of this task is to automatically identify duplicate products using AI embeddings and generate a clean catalog containing only unique products.

# 🎯 Objective

Develop an AI-powered system that:

- Loads the fashion product dataset.
- Generates semantic embeddings for products.
- Detects duplicate products.
- Creates a unique product catalog.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

print("="*50)
print("DEVICE CONFIGURATION")
print("="*50)
print("Device :", device)

if device == "cuda":
    print("GPU :", torch.cuda.get_device_name(0))

DEVICE CONFIGURATION
Device : cuda
GPU : Tesla T4


# 📂 Dataset

Dataset:
Fashion Product Images Dataset

Contents

- styles.csv
- images/

Total Products

44,000+

In [3]:
dataset_path="/kaggle/input/datasets/paramaggarwal/fashion-product-images-dataset/fashion-dataset"

styles_path=os.path.join(dataset_path,"styles.csv")

df=pd.read_csv(styles_path,on_bad_lines="skip")

print(df.shape)

df.head()

(44424, 10)


,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName
0,15970,Men,Apparel,Topwear,Shirts,Navy Blue,Fall,2011.0,Casual,Turtle Check Men Navy Blue Shirt
1,39386,Men,Apparel,Bottomwear,Jeans,Blue,Summer,2012.0,Casual,Peter England Men Party Blue Jeans
2,59263,Women,Accessories,Watches,Watches,Silver,Winter,2016.0,Casual,Titan Women Silver Watch
3,21379,Men,Apparel,Bottomwear,Track Pants,Black,Fall,2011.0,Casual,Manchester United Men Solid Black Track Pants
4,53759,Men,Apparel,Topwear,Tshirts,Grey,Summer,2012.0,Casual,Puma Men Grey T-shirt


In [4]:
df=df[
[
"id",
"productDisplayName",
"masterCategory",
"subCategory",
"articleType",
"baseColour",
"gender",
"season",
"usage"
]
]

df=df.dropna()

print(df.shape)

(44077, 9)


In [5]:
print("Products :",len(df))

print("Article Types :",df["articleType"].nunique())

print("Categories :",df["masterCategory"].nunique())

df.head()

Products : 44077
Article Types : 142
Categories : 7


,id,productDisplayName,masterCategory,subCategory,articleType,baseColour,gender,season,usage
0,15970,Turtle Check Men Navy Blue Shirt,Apparel,Topwear,Shirts,Navy Blue,Men,Fall,Casual
1,39386,Peter England Men Party Blue Jeans,Apparel,Bottomwear,Jeans,Blue,Men,Summer,Casual
2,59263,Titan Women Silver Watch,Accessories,Watches,Watches,Silver,Women,Winter,Casual
3,21379,Manchester United Men Solid Black Track Pants,Apparel,Bottomwear,Track Pants,Black,Men,Fall,Casual
4,53759,Puma Men Grey T-shirt,Apparel,Topwear,Tshirts,Grey,Men,Summer,Casual


# 🤖 AI Model

Sentence Transformer converts product descriptions into semantic embeddings.

Products with highly similar embeddings are considered duplicates.

In [6]:
model=SentenceTransformer("all-MiniLM-L6-v2")

print("Sentence Transformer Loaded Successfully")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Sentence Transformer Loaded Successfully


In [7]:
df["product_text"]=(
df["productDisplayName"]+" "+
df["masterCategory"]+" "+
df["subCategory"]+" "+
df["articleType"]+" "+
df["baseColour"]+" "+
df["gender"]+" "+
df["usage"]
)

df[["productDisplayName","product_text"]].head()

,productDisplayName,product_text
0,Turtle Check Men Navy Blue Shirt,Turtle Check Men Navy Blue Shirt Apparel Topwe...
1,Peter England Men Party Blue Jeans,Peter England Men Party Blue Jeans Apparel Bot...
2,Titan Women Silver Watch,Titan Women Silver Watch Accessories Watches W...
3,Manchester United Men Solid Black Track Pants,Manchester United Men Solid Black Track Pants ...
4,Puma Men Grey T-shirt,Puma Men Grey T-shirt Apparel Topwear Tshirts ...


In [8]:
catalog=df.head(2000).copy()

embeddings=model.encode(
catalog["product_text"].tolist(),
batch_size=64,
show_progress_bar=True,
convert_to_numpy=True
)

print(embeddings.shape)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

(2000, 384)


In [9]:
similarity=cosine_similarity(embeddings)

print(similarity.shape)

(2000, 2000)


In [10]:
threshold=0.90

visited=set()

groups=[]

for i in range(len(catalog)):

    if i in visited:
        continue

    group=[i]

    visited.add(i)

    for j in range(i+1,len(catalog)):

        if similarity[i][j]>=threshold:

            group.append(j)

            visited.add(j)

    groups.append(group)

print("Groups :",len(groups))

Groups : 1292


In [11]:
unique=[]

for group in groups:

    unique.append(catalog.iloc[group[0]])

unique_catalog=pd.DataFrame(unique)

print("Original :",len(catalog))

print("Unique :",len(unique_catalog))

Original : 2000
Unique : 1292


In [12]:
unique_catalog.to_csv(
"unique_product_catalog.csv",
index=False
)

print("Catalog Saved Successfully")

Catalog Saved Successfully


In [13]:
unique_catalog.head(20)

,id,productDisplayName,masterCategory,subCategory,articleType,baseColour,gender,season,usage,product_text
0,15970,Turtle Check Men Navy Blue Shirt,Apparel,Topwear,Shirts,Navy Blue,Men,Fall,Casual,Turtle Check Men Navy Blue Shirt Apparel Topwe...
1,39386,Peter England Men Party Blue Jeans,Apparel,Bottomwear,Jeans,Blue,Men,Summer,Casual,Peter England Men Party Blue Jeans Apparel Bot...
2,59263,Titan Women Silver Watch,Accessories,Watches,Watches,Silver,Women,Winter,Casual,Titan Women Silver Watch Accessories Watches W...
3,21379,Manchester United Men Solid Black Track Pants,Apparel,Bottomwear,Track Pants,Black,Men,Fall,Casual,Manchester United Men Solid Black Track Pants ...
4,53759,Puma Men Grey T-shirt,Apparel,Topwear,Tshirts,Grey,Men,Summer,Casual,Puma Men Grey T-shirt Apparel Topwear Tshirts ...
5,1855,Inkfruit Mens Chain Reaction T-shirt,Apparel,Topwear,Tshirts,Grey,Men,Summer,Casual,Inkfruit Mens Chain Reaction T-shirt Apparel T...
6,30805,Fabindia Men Striped Green Shirt,Apparel,Topwear,Shirts,Green,Men,Summer,Ethnic,Fabindia Men Striped Green Shirt Apparel Topwe...
7,26960,Jealous 21 Women Purple Shirt,Apparel,Topwear,Shirts,Purple,Women,Summer,Casual,Jealous 21 Women Purple Shirt Apparel Topwear ...
8,29114,Puma Men Pack of 3 Socks,Accessories,Socks,Socks,Navy Blue,Men,Summer,Casual,Puma Men Pack of 3 Socks Accessories Socks Soc...
9,30039,Skagen Men Black Watch,Accessories,Watches,Watches,Black,Men,Winter,Casual,Skagen Men Black Watch Accessories Watches Wat...


# 🏗 Workflow

```text
Fashion Dataset
       │
       ▼
Product Metadata
       │
       ▼
Sentence Transformer
       │
       ▼
Semantic Embeddings
       │
       ▼
Cosine Similarity
       │
       ▼
Duplicate Detection
       │
       ▼
Unique Product Catalog
```

# ✅ Conclusion

An AI-powered semantic duplicate detection system was developed using Sentence Transformer embeddings.

The system successfully generated semantic embeddings, detected duplicate products, grouped similar products, and created a clean unique product catalog suitable for e-commerce applications.